# Day 4: Sequential Logic Fundamentals

## Pre-Class Videos (~50 minutes total)

| # | Segment | Duration | File | Slides |
|---|---------|----------|------|--------|
| 1 | Clocks and Edge-Triggered Behavior | ~12 min | `d04_s1_clocks_and_edges.html` | 6 |
| 2 | Nonblocking Assignment | ~15 min | `d04_s2_nonblocking_assignment.html` | 6 |
| 3 | Flip-Flops With Reset and Enable | ~10 min | `d04_s3_flip_flop_variants.html` | 5 |
| 4 | Counters and Clock Division | ~13 min | `d04_s4_counters_and_clock_division.html` | 8 |

## Code Examples

| File | Description | Synthesizable? |
|------|-------------|----------------|
| `code/day04_ex01_d_flip_flop.v` | D flip-flop with sync reset and enable | Yes |
| `code/day04_ex02_led_blinker.v` | Counter-based 1 Hz LED blinker | Yes |
| `code/day04_ex03_shift_register_demo.v` | Side-by-side blocking vs nonblocking | Sim only |

## Diagrams

| File | Used In | Description |
|------|---------|-------------|
| `diagrams/d04_pipeline_blocking.svg` | Seg 2 | Blocking vs nonblocking pipeline comparison |

## Pre-Class Quiz

See `day04_quiz.md` — 4 questions. Also embedded at end of Segment 4.

## Directory Structure

```
week1_day04/
├── day04_readme.md
├── day04_quiz.md
├── d04_s1_clocks_and_edges.html
├── d04_s2_nonblocking_assignment.html
├── d04_s3_flip_flop_variants.html
├── d04_s4_counters_and_clock_division.html
├── code/
│   ├── day04_ex01_d_flip_flop.v
│   ├── day04_ex02_led_blinker.v
│   └── day04_ex03_shift_register_demo.v
└── diagrams/
    └── d04_pipeline_blocking.svg
```

---
## Code Examples

### `day04_ex01_d_flip_flop.v`

```verilog
//-----------------------------------------------------------------------------
// File:    day04_ex01_d_flip_flop.v
// Course:  Accelerated HDL for Digital System Design — Day 4
//
// Description:
//   D flip-flop with synchronous reset and optional clock enable.
//   This is the fundamental sequential building block.
//
// Simulate:
//   iverilog -o d_ff_tb day04_ex01_d_flip_flop.v tb_d_ff.v
//   vvp d_ff_tb
//   gtkwave d_ff_tb.vcd
//-----------------------------------------------------------------------------
module d_flip_flop (
    input  wire i_clk,
    input  wire i_reset,
    input  wire i_enable,
    input  wire i_d,
    output reg  o_q
);
    always @(posedge i_clk) begin
        if (i_reset)
            o_q <= 1'b0;
        else if (i_enable)
            o_q <= i_d;
        // else: o_q holds — NOT a latch (this is sequential)
    end
endmodule
```

### `day04_ex02_led_blinker.v`

```verilog
//-----------------------------------------------------------------------------
// File:    day04_ex02_led_blinker.v
// Course:  Accelerated HDL for Digital System Design — Day 4
// Board:   Nandland Go Board (Lattice iCE40 HX1K, VQ100)
//
// Description:
//   Counter-based 1 Hz LED blinker. Divides 25 MHz clock to ~1 Hz
//   by counting to 12,500,000 per half-period, then toggling.
//   Also demonstrates MSB-tap trick for multi-speed blink.
//
// Build:
//   yosys -p "synth_ice40 -top led_blinker -json day04_ex02_led_blinker.json" day04_ex02_led_blinker.v
//   nextpnr-ice40 --hx1k --package vq100 --pcf go_board.pcf \
//                 --json day04_ex02_led_blinker.json --asc day04_ex02_led_blinker.asc
//   icepack day04_ex02_led_blinker.asc day04_ex02_led_blinker.bin
//   iceprog day04_ex02_led_blinker.bin
//-----------------------------------------------------------------------------
module led_blinker (
    input  wire i_clk,       // 25 MHz
    output wire o_led1,      // ~1 Hz  (active low)
    output wire o_led2,      // ~1.5 Hz (counter bit tap)
    output wire o_led3,      // ~3 Hz
    output wire o_led4       // ~6 Hz
);
    // --- 1 Hz toggle via explicit count ---
    reg [23:0] r_counter = 24'd0;
    reg        r_led     = 1'b0;

    always @(posedge i_clk) begin
        if (r_counter == 24'd12_499_999) begin
            r_counter <= 24'd0;
            r_led     <= ~r_led;
        end else begin
            r_counter <= r_counter + 1;
        end
    end

    assign o_led1 = ~r_led;   // active low

    // --- Multi-speed via MSB taps (approximate) ---
    reg [23:0] r_free = 24'd0;
    always @(posedge i_clk)
        r_free <= r_free + 1;

    assign o_led2 = ~r_free[23];  // ~1.5 Hz
    assign o_led3 = ~r_free[22];  // ~3 Hz
    assign o_led4 = ~r_free[21];  // ~6 Hz
endmodule
```

### `day04_ex03_shift_register_demo.v`

```verilog
//-----------------------------------------------------------------------------
// File:    day04_ex03_shift_register_demo.v
// Course:  Accelerated HDL for Digital System Design — Day 4
//
// Description:
//   Side-by-side comparison of blocking vs. nonblocking assignment
//   in a 3-stage shift register. Simulate both and compare waveforms.
//
// Simulate:
//   iverilog -o shift_demo day04_ex03_shift_register_demo.v
//   vvp shift_demo
//   gtkwave shift_demo.vcd
//-----------------------------------------------------------------------------
module shift_blocking (
    input  wire i_clk, i_d,
    output reg  o_q
);
    reg r_a, r_b;
    always @(posedge i_clk) begin
        r_a = i_d;    // blocking: r_a gets i_d NOW
        r_b = r_a;    // r_b gets UPDATED r_a (= i_d)
        o_q = r_b;    // o_q gets UPDATED r_b (= i_d)
    end               // Result: all three get i_d — NO pipeline!
endmodule

module shift_nonblocking (
    input  wire i_clk, i_d,
    output reg  o_q
);
    reg r_a, r_b;
    always @(posedge i_clk) begin
        r_a <= i_d;   // scheduled: r_a ← i_d(current)
        r_b <= r_a;   // scheduled: r_b ← r_a(current, old value)
        o_q <= r_b;   // scheduled: o_q ← r_b(current, old value)
    end               // Result: data shifts through — 3-cycle delay
endmodule
```

---
## Pre-Class Self-Check Quiz

**Q1:** What happens if you use `=` instead of `<=` in `always @(posedge clk)` with two registers?

<details><summary>Answer</summary>
Both registers get the same value in one cycle. With `<=`, both right-hand sides are evaluated first with current values, then all updates happen simultaneously.
</details>

**Q2:** Is a missing `else` in `always @(posedge clk)` a latch?

<details><summary>Answer</summary>
No! Flip-flops inherently hold their value between clock edges. Latch inference only occurs in combinational `always @(*)` blocks.
</details>

**Q3:** Write a D flip-flop with synchronous reset from memory.

<details><summary>Answer</summary>
```verilog
always @(posedge i_clk)
    if (i_reset) r_q <= 1'b0;
    else         r_q <= r_d;
```
</details>

**Q4:** You want an LED to blink at ~2 Hz from a 25 MHz clock. What counter value do you count to?

<details><summary>Answer</summary>
2 Hz = toggle every 0.25s = 6,250,000 clock cycles per half-period. Count to 6,250,000 - 1, then toggle and reset.
</details>